In [30]:
import shutil
import pandas as pd
from pathlib import Path


In [31]:
PROJECT_ROOT = Path.cwd()

DATA_ROOT = PROJECT_ROOT / "Data" / "Balabit-dataset"

TEST_SRC = DATA_ROOT / "test_files"
PROTOCOL1_DST = DATA_ROOT / "testing_files_protocol1"
PROTOCOL2_DST = DATA_ROOT / "testing_files_protocol2"

LABEL_CSV = DATA_ROOT / "public_labels.csv"



In [32]:
labels_df = pd.read_csv(LABEL_CSV)

# 按你的定义：
# 0 = imposter
# 1 = genuine
session2label = dict(
    zip(labels_df["filename"], labels_df["is_illegal"])
)

print(f"Loaded {len(session2label)} labeled sessions")




Loaded 816 labeled sessions


In [33]:
labels_df["is_illegal"].value_counts()



is_illegal
0    411
1    405
Name: count, dtype: int64

In [ ]:
def build_testing_protocol1():
    if PROTOCOL1_DST.exists():
        shutil.rmtree(PROTOCOL1_DST)
    PROTOCOL1_DST.mkdir(parents=True)

    kept = 0

    for user_dir in sorted(TEST_SRC.iterdir()):
        if not user_dir.is_dir():
            continue

        user_out = PROTOCOL1_DST / user_dir.name
        user_out.mkdir(parents=True)

        for session_path in user_dir.iterdir():
            if not session_path.is_file():
                continue

            session_name = session_path.name

            # only save sessions in public label list
            if session_name not in session2label:
                continue

            # Protocol 1: only genuine (is_illegal == 0)
            if session2label[session_name] == 0:
                shutil.copy(session_path, user_out / session_name)
                kept += 1

    print(f"[Protocol 1] Finished. Kept {kept} genuine sessions.")


In [35]:
build_testing_protocol1()

[Protocol 1] Finished. Kept 411 genuine sessions.


In [ ]:
def build_testing_protocol2():
    if PROTOCOL2_DST.exists():
        shutil.rmtree(PROTOCOL2_DST)
    PROTOCOL2_DST.mkdir(parents=True)

    genuine_cnt = 0
    imposter_cnt = 0

    for user_dir in sorted(TEST_SRC.iterdir()):
        if not user_dir.is_dir():
            continue

        user_root = PROTOCOL2_DST / user_dir.name
        genuine_dir = user_root / "genuine"
        imposter_dir = user_root / "imposter"

        genuine_dir.mkdir(parents=True)
        imposter_dir.mkdir(parents=True)

        for session_path in user_dir.iterdir():
            if not session_path.is_file():
                continue

            session_name = session_path.name

            # only save sessions in public label list
            if session_name not in session2label:
                continue

            is_illegal = session2label[session_name]

            # 0 = genuine, 1 = imposter
            if is_illegal == 0:
                shutil.copy(session_path, genuine_dir / session_name)
                genuine_cnt += 1
            else:
                shutil.copy(session_path, imposter_dir / session_name)
                imposter_cnt += 1

    print("[Protocol 2] Finished.")
    print(f"  Genuine : {genuine_cnt}")
    print(f"  Imposter: {imposter_cnt}")


In [37]:
build_testing_protocol2()


[Protocol 2] Finished.
  Genuine : 411
  Imposter: 405
